# Summarising results

This notebook summarises results into a Pandas dataframe which is then
reformatted into a table suitable for publishing.


In [1]:
import json
from collections import defaultdict
from collections.abc import Mapping
from datetime import datetime, timezone
from itertools import product
from pathlib import Path
from typing import Any, Iterable, Literal, TypeAlias, cast

import pandas as pd
from pandas.io.formats.style import Styler

import qaoa_parameter_setting.utils as utils
from qaoa_parameter_setting.utils.database import ResultsDatabase
from qaoa_parameter_setting.utils.database.summary_table_formatter import (
    convert_evaluation_to_multicolumn_latex,
    formatted_styler_for,
)
from qaoa_parameter_setting.utils.types import (
    Depth,
    EvaluationType,
    GraphKey,
    GraphType,
    MethodConfigJSON,
)

In [2]:
# Make sure to change this when regenerating all tables.
PROBLEM_CLASS: Literal["MC", "MIS"] = "MIS"

In [3]:
# TABLE_JSON: str | None = f"summary_tables_{PROBLEM_CLASS}.json"
TABLE_JSON: str | None = None
table = ResultsDatabase(TABLE_JSON, problem_class=PROBLEM_CLASS)

## Setup table with training data and min-max cuts.


In [4]:
if TABLE_JSON is None:

    def ignore_results(filename: str, results: dict[str, Any]) -> bool:
        """Ignore Parameter Transfer results for the tables."""
        if "PT_" in filename:
            return True

        # Ignore Fixed Angles if we are creating the MIS table.
        if PROBLEM_CLASS == "MIS" and "FA_" in filename:
            return True
        return False

    # Add training data. Only the "best" results are kept, per graph, trainer config
    # file, and depth.
    for path in [
        "../data/training/random_regular",
        "../data/training/heavy_hex",
        "../data/training/line_to_full",
        "../data/training/erdos_renyi",
    ]:
        table.add_data(path, ignore_file_function=ignore_results)

In [5]:
if TABLE_JSON is None and table.problem_class == "MC":
    # Add min- and max-cut data. If some graph instances do not have a minmax_cuts
    # entry, an error will be thrown later.
    table.add_minmax_cut_data("../data/minmax_cuts/random_regular")
    table.add_minmax_cut_data("../data/minmax_cuts/heavy_hex")
    table.add_minmax_cut_data("../data/minmax_cuts/line_to_full")
    table.add_minmax_cut_data("../data/minmax_cuts/erdos_renyi")

## Identify missing min- and max-cuts data


In [6]:
if table.problem_class == "MC":
    missing_minmax_cuts = table.missing_minmax_cuts()
    if len(missing_minmax_cuts) == 0:
        print("All minmax_cuts data accounted for.")
    else:
        print(
            "The following graphs are missing min- and max-cuts data. "
            + "Generate them with compute_min_max_for_graph.py"
        )
        for _graph in missing_minmax_cuts:
            print("- {}".format(_graph))
else:
    print(f"Min-/Max-cut data not necessary for {table.problem_class} data.")

Min-/Max-cut data not necessary for MIS data.


## List of methods per evaluation type


In [7]:
table.print_methods_by_evaluation()

      MPS (Aer)        |     MPS (Quimb)     |         PP         |          SV         
----------------------------------------------------------------------------------------
F_MPSAer.json          | F_MPS.json          | F_PP.json          | F_SV.json           
I_MPSAer.json          | I_MPS.json          | I_PP.json          | I_SV.json           
TQA_MPSAer_no_opt.json | TQA_MPS_no_opt.json | TQA_PP_no_opt.json | LR_SV_angle_opt.json
TQA_MPSAer_opt.json    | TQA_MPS_opt.json    | TQA_PP_opt.json    | LR_SV_opt.json      
                       |                     |                    | TQA_SV_no_opt.json  
                       |                     |                    | TQA_SV_opt.json     
                       |                     |                    | TS_SV.json          


## Get raw data table


### Define target instances


In [8]:
# Define target instances if we are filtering.
TARGET_INSTANCES: dict[EvaluationType, set[GraphKey] | dict[bool, set[GraphKey]]] | None
"""The target instances for all Summary Tables of this problem class.

We only filter instances here for MIS tables. MAXCUT filtering is done during
table generation using `only_common_instances`.
"""
if table.problem_class == "MIS":
    LARGE_INSTANCES: set[GraphKey] = set(
        [
            _graph_format.format(idx)
            for idx in range(10)
            for _graph_format in [
                "{:03}_100nodes_random3regular.json",
                "{:03}_7_3_heavyhex_144nodes_weighted.json",
                "{:03}_100nodes_2swap_layers.json",
                "{:03}_70nodes_erdosrenyi20percent.json",
            ]
        ]
    )
    SMALL_INSTANCES = {
        # 10x 20-node 20-percent Erdös-Rényi graphs
        "000_20nodes_erdosrenyi20percent.json",
        "001_20nodes_erdosrenyi20percent.json",
        "002_20nodes_erdosrenyi20percent.json",
        "003_20nodes_erdosrenyi20percent.json",
        "004_20nodes_erdosrenyi20percent.json",
        "005_20nodes_erdosrenyi20percent.json",
        "006_20nodes_erdosrenyi20percent.json",
        "007_20nodes_erdosrenyi20percent.json",
        "008_20nodes_erdosrenyi20percent.json",
        "009_20nodes_erdosrenyi20percent.json",
        # 10x 1-by-2 21-node Heavy-Hex graphs
        "000_1_2_heavyhex_21nodes_weighted.json",
        "001_1_2_heavyhex_21nodes_weighted.json",
        "002_1_2_heavyhex_21nodes_weighted.json",
        "003_1_2_heavyhex_21nodes_weighted.json",
        "004_1_2_heavyhex_21nodes_weighted.json",
        "005_1_2_heavyhex_21nodes_weighted.json",
        "006_1_2_heavyhex_21nodes_weighted.json",
        "007_1_2_heavyhex_21nodes_weighted.json",
        "008_1_2_heavyhex_21nodes_weighted.json",
        "009_1_2_heavyhex_21nodes_weighted.json",
        # 1x 20-node line-to-full graph per 1 to 10 swap layers, for total of 10x graphs
        "001_20nodes_1swap_layers.json",
        "001_20nodes_2swap_layers.json",
        "001_20nodes_3swap_layers.json",
        "001_20nodes_4swap_layers.json",
        "001_20nodes_5swap_layers.json",
        "001_20nodes_6swap_layers.json",
        "001_20nodes_7swap_layers.json",
        "001_20nodes_8swap_layers.json",
        "001_20nodes_9swap_layers.json",
        "001_20nodes_10swap_layers.json",
        # 1x 20-node random-regular graph per degree d=3,4,5,6,7,8,9 for a total of 7x graphs
        "001_20nodes_random3regular.json",
        "001_20nodes_random4regular.json",
        "001_20nodes_random5regular.json",
        "001_20nodes_random6regular.json",
        "001_20nodes_random7regular.json",
        "001_20nodes_random8regular.json",
        "001_20nodes_random9regular.json",
    }

    TARGET_INSTANCES = {
        "MPS": {
            False: LARGE_INSTANCES,
            True: LARGE_INSTANCES,
        },
        "PP": LARGE_INSTANCES,
        "SV": SMALL_INSTANCES,
    }
else:
    TARGET_INSTANCES = None

### Define reference methods for filtering runs


In [9]:
# Determine reference methods for depth 10 tables.
REFERENCE_METHODS_10: (
    dict[
        EvaluationType | tuple[Literal["MPS"], bool],
        dict[
            GraphType,
            MethodConfigJSON,
        ],
    ]
    | None
)
"""Reference methods for depth 10 tables, only used for MAXCUT data.

These were found by hand, identifying methods with enough runs but that also
reduced the number of additional simulations to run.
"""
if table.problem_class == "MC":
    REFERENCE_METHODS_10 = {
        ("MPS", False): {
            "erdos_renyi": "F_MPS.json",
            "heavy_hex": "F_MPS.json",
            "line_to_full": "LR_MPS_opt.json",
            "random_regular": "FA_MPS_opt.json",
        },
        ("MPS", True): {
            "erdos_renyi": "F_MPSAer.json",
            "heavy_hex": "F_MPSAer.json",
            "line_to_full": "LR_MPSAer_opt.json",
            "random_regular": "FA_MPSAer_opt.json",
        },
        "SV": {
            "erdos_renyi": "TQA_SV_opt.json",
            "heavy_hex": "F_SV.json",
            "line_to_full": "LR_SV_opt.json",
            "random_regular": "TQA_SV_opt.json",
        },
        "PP": {
            "erdos_renyi": "TQA_PP_opt.json",
            "heavy_hex": "TQA_PP_opt.json",
            "line_to_full": "TQA_PP_opt.json",
            "random_regular": "TQA_PP_opt.json",
        },
    }  # type: ignore[assignment]
elif table.problem_class == "MIS":
    REFERENCE_METHODS_10 = None
else:
    raise NotImplementedError(
        "Cannot determine reference methods for problem class {!r}.".format(
            table.problem_class
        )
    )

### Define number of nodes for filtering runs


In [10]:
NUM_NODES: Iterable[int] | dict[str, Iterable[int] | dict[bool, Iterable[int]]] | None
if table.problem_class == "MIS":
    NUM_NODES = {
        "MPS": {True: [70, 100, 144], False: [70, 100, 144]},
        "PP": [70, 100, 144],
        "SV": [20, 21],
    }
else:
    NUM_NODES = None

In [11]:
# Only select results with the best energy per config and instance:
table = table.only_best_parameters("config")

if TARGET_INSTANCES is not None or NUM_NODES is not None:
    table = table.filter_by(
        instance_filter=TARGET_INSTANCES,
        num_nodes=NUM_NODES,
    )


# This is the _raw_ table with all results
df: pd.DataFrame = table.to_dataframe()
df

,instance,num_nodes,graph_type,trainer_config,method,depth,energy,trainer,evaluation,evaluation_label,...,metadata,result_index,run_datetime,result_key_index,approximation_ratio,mps_bond_dimension,mps_threshold,pp_max_weight,pp_min_abs_coeff,fa_degree
0,001_20nodes_random3regular.json,20,random_regular,TQA_SV_no_opt.json,TQA_no_opt.json,10,32.762930,TQATrainer,SV,SV,...,"{'iteration': '0', 'version': 25, 'evaluator':...",0,2026-01-19 21:00:56,0.0,NaN,NaN,NaN,NaN,NaN,None
1,001_20nodes_random3regular.json,20,random_regular,TQA_SV_no_opt.json,TQA_no_opt.json,1,18.063329,TQATrainer,SV,SV,...,"{'iteration': '0', 'version': 25, 'evaluator':...",0,2026-01-20 18:26:24,0.0,NaN,NaN,NaN,NaN,NaN,None
2,001_20nodes_random3regular.json,20,random_regular,TQA_SV_no_opt.json,TQA_no_opt.json,2,29.183569,TQATrainer,SV,SV,...,"{'iteration': '0', 'version': 25, 'evaluator':...",0,2026-01-20 18:52:25,0.0,NaN,NaN,NaN,NaN,NaN,None
3,001_20nodes_random3regular.json,20,random_regular,TQA_SV_no_opt.json,TQA_no_opt.json,3,32.443083,TQATrainer,SV,SV,...,"{'iteration': '0', 'version': 25, 'evaluator':...",0,2026-01-20 19:42:50,0.0,NaN,NaN,NaN,NaN,NaN,None
4,001_20nodes_random3regular.json,20,random_regular,TQA_SV_no_opt.json,TQA_no_opt.json,4,32.591822,TQATrainer,SV,SV,...,"{'iteration': '0', 'version': 25, 'evaluator':...",0,2026-01-20 20:03:57,0.0,NaN,NaN,NaN,NaN,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3051,009_20nodes_erdosrenyi20percent.json,20,erdos_renyi,TS_SV.json,TS.json,6,38.471448,TransitionStatesTrainer,SV,SV,...,"{'iteration': '6', 'version': 33, 'evaluator':...",0,2026-04-28 19:02:35,6.2,NaN,NaN,NaN,NaN,NaN,None
3052,009_20nodes_erdosrenyi20percent.json,20,erdos_renyi,TS_SV.json,TS.json,7,38.942041,TransitionStatesTrainer,SV,SV,...,"{'iteration': '7', 'version': 33, 'evaluator':...",0,2026-04-28 19:02:35,7.2,NaN,NaN,NaN,NaN,NaN,None
3053,009_20nodes_erdosrenyi20percent.json,20,erdos_renyi,TS_SV.json,TS.json,8,39.073265,TransitionStatesTrainer,SV,SV,...,"{'iteration': '8', 'version': 33, 'evaluator':...",0,2026-04-28 19:02:35,8.2,NaN,NaN,NaN,NaN,NaN,None
3054,009_20nodes_erdosrenyi20percent.json,20,erdos_renyi,TS_SV.json,TS.json,9,39.074932,TransitionStatesTrainer,SV,SV,...,"{'iteration': '9', 'version': 33, 'evaluator':...",0,2026-04-28 19:02:35,9.2,NaN,NaN,NaN,NaN,NaN,None


## Get formatted and pivoted table and styler

With Pandas, dataframes are formatted with Stylers.
`qaoa_parameter_setting.utils.database.summary_table_formatter.formatted_styler_for` automatically
stylises the Styler and returns the pivoted dataframe and Styler.


### Example summary table with `"text"` formatting

The following cells format the summary results into a single table suitable for
Jupyter notebooks. Later cells will format the table for LaTeX and save them to
a file.

As we can generate tables for both MAXCUT and MIS, we use `performance_metric`
to store the field name used to quantify performance.


In [12]:
performance_metric: Literal["approximation_ratio", "energy"]
if table.problem_class == "MC":
    performance_metric = "approximation_ratio"
elif table.problem_class == "MIS":
    performance_metric = "energy"
else:
    raise NotImplementedError(
        "Performance summary tables not implemented for {!r} problem class.".format(
            table.problem_class
        )
    )

#### MAXCUT Approximation Ratio or MIS Normalized Energy


In [13]:
pivot, styler, _ = formatted_styler_for(
    table,
    depths=10,
    agg_values=performance_metric,
    with_fancy_values=True,
    cmap={"MPS (Aer)": "BuPu", "MPS (Quimb)": "YlOrBr", "PP": "YlGn", "SV": "PuBu"},
    precision=2,
    missing_data_str="-",
    target_format="text",
    show_empty_rows=True,
    # exclude_methods=["RTS", "TS"],
)
styler

#### Number of instances and nodes


In [14]:
pivot, styler, _ = formatted_styler_for(
    table,
    depths=10,
    num_nodes=None,
    agg_values="num_instances",
    with_fancy_values=True,
    cmap="Greens",
    precision=0,
    missing_data_str="-",
    target_format="text",
    show_empty_rows=True,
    # exclude_methods=["RTS", "TS"],
)
styler

## Combined table


In [15]:
table_cmap = {
    (k1, k2): v if k2 == performance_metric else "RdPu"
    for k1, v in {
        "MPS (Aer)": "BuPu",
        "MPS (Quimb)": "YlOrBr",
        "PP": "YlGn",
        "SV": "PuBu",
    }.items()
    for k2 in [performance_metric, "num_instances"]
}
pivot, styler, cmap_ranges = formatted_styler_for(
    table,
    depths=10,
    agg_values=[performance_metric, "num_instances"],
    with_fancy_values=True,
    cmap=table_cmap,
    precision={performance_metric: 2, "num_instances": 0},
    missing_data_str="-",
    target_format="text",
    show_empty_rows=False,
    # exclude_methods=["RTS"],
)
styler

In [16]:
table_cmap = {
    (k1, k2): v if k2 == performance_metric else "RdPu"
    for k1, v in {
        "MPS (Aer)": "BuPu",
        "MPS (Quimb)": "YlOrBr",
        "PP": "YlGn",
        "SV": "PuBu",
    }.items()
    for k2 in [performance_metric, "num_instances"]
}
pivot, styler, cmap_ranges = formatted_styler_for(
    table,
    depths=6,
    num_nodes={
        "MPS (Aer)": [70, 100],
        "MPS (Quimb)": [70, 100],
        "PP": [70, 100],
        "SV": 20,
    },
    agg_values=[performance_metric, "num_instances"],
    with_fancy_values=True,
    cmap=table_cmap,
    precision={performance_metric: 2, "num_instances": 0},
    missing_data_str="-",
    target_format="text",
    # exclude_methods=["RTS"],
)
styler

## Create and save tables for LaTeX

Here we format tables into LaTeX files and save them into separate `.tex` files
suitable for inclusion in a paper. Files are saved with the current date and
time for tracking changes. These can be compiled into a preview of all tables
with `pdflatex summary_tables.tex`


In [17]:
AggValue: TypeAlias = Literal["num_instances", "approximation_ratio", "energy"]
values_to_plot: list[
    tuple[AggValue | Literal["both"], AggValue | list[AggValue], str, str]
] = [
    (
        "num_instances",
        "num_instances",
        "Number of Instances (num. nodes in brackets)",
        " Cells are colored based on the number of instances for the given configuration, with darker colors indicating more instances.",
    )
]
# Define values and captions for tables
if table.problem_class == "MC":
    values_to_plot.extend(
        [
            (
                "approximation_ratio",
                "approximation_ratio",
                r"Avg. Approx. Ratio $\pm$ standard deviation in percentage for MAXCUT",
                " Evaluation methods are shown in different colors, with darker colors indicating better approximation ratios.",
            ),
            (
                "both",
                ["approximation_ratio", "num_instances"],
                "Avg. Approx. Ratio and Number of Instances for Various Evaluation Methods for MAXCUT",
                " Evaluation methods are shown in different colors for approximation ratios."
                + " Darker colors indicating better approximation ratios or more instances."
                + " Approximation ratios are in percentage, with $\\pm$ their standard deviation."
                + " The range of graph sizes are shown in brackets, next to the number of instances.",
            ),
        ]
    )
elif table.problem_class == "MIS":
    values_to_plot.extend(
        [
            (
                "energy",
                "energy",
                r"Avg. Energy $\pm$ standard deviation for MIS",
                " Evaluation methods are shown in different colors, with darker colors indicating better energies."
                + " The minimum and maximum energies defining the range of colors are shared between MPS and PP."
                + " Note that energies include invalid solutions, which are penalized by the Hamiltonian.",
            ),
            (
                "both",
                ["energy", "num_instances"],
                "Avg. Energy and Number of Instances for Various Evaluation Methods for MIS",
                " Evaluation methods are shown in different colors for energies."
                + " Darker colors indicating better energies or more instances."
                + " The minimum and maximum energies defining the range of colors are shared between MPS and PP."
                + " Average energies are shown, with $\\pm$ their standard deviation."
                + " The range of graph sizes are shown in brackets, next to the number of instances.",
            ),
        ]
    )

In [18]:
## Create the CMAPs for each table.
TABLE_CMAPS_PER_EVALUATION = {
    "MPS (Aer)": "BuPu",
    "MPS (Quimb)": "YlOrBr",
    "PP": "YlGn",
    "SV": "PuBu",
}
table_cmap_num_instances = "Greens"
# The combined tables use different ranges for SV. MAXCUT has the same range for all evaluation
# methods, whereas SV has its own range for MIS data.
if table.problem_class == "MC":
    table_cmap_performance = TABLE_CMAPS_PER_EVALUATION
    table_cmap_combined = [
        {
            (str(k1), performance_metric): v
            for k1, v in TABLE_CMAPS_PER_EVALUATION.items()
        },
        {
            (str(k1), "num_instances"): "RdPu"
            for k1 in TABLE_CMAPS_PER_EVALUATION.keys()
        },
    ]
else:
    table_cmap_performance = [
        {"MPS (Aer)": "BuPu", "MPS (Quimb)": "YlOrBr", "PP": "YlGn"},
        {"SV": "PuBu"},
    ]
    table_cmap_combined = [
        {
            (str(k1), performance_metric): v
            for k1, v in TABLE_CMAPS_PER_EVALUATION.items()
            if k1 != "SV"
        },
        {
            (str(k1), performance_metric): v
            for k1, v in TABLE_CMAPS_PER_EVALUATION.items()
            if k1 == "SV"
        },
        {
            (str(k1), "num_instances"): "RdPu"
            for k1 in TABLE_CMAPS_PER_EVALUATION.keys()
        },
    ]

In [19]:
now = datetime.now(tz=timezone.utc)
generated_on_str = "% Generated on {now:%Y-%m-%d} at {now:%H:%M:%S} UTC\n".format(
    now=now
)
# Open a summary list_of_tables file for previewing all tables.
with open("list_of_tables_{}.tex".format(table.problem_class), "w") as f:
    # Write date and time to list_of_tables
    f.write(generated_on_str)

    # Iterate over all depths, sorted so they're included in increasing order in
    # list_of_tables
    for depth in sorted(int(x) for x in table.to_dataframe()["depth"].unique()):
        # Write a section title.
        _ = f.write("\n\\section{{Tables for depth $P={}$}}\n".format(depth))
        for filename_suffix, values, value_label, additional_caption in values_to_plot:
            # This is the filename for this table.
            _latex_filename = "table_{problem_class}_p{depth:02}_{suffix}.tex".format(
                problem_class=table.problem_class,
                depth=int(depth),
                suffix=filename_suffix,
            )
            if REFERENCE_METHODS_10 is not None and depth == 10:
                sub_table = table.only_common_instances(
                    REFERENCE_METHODS_10, per_depth=True
                )
            else:
                sub_table = table
            # Get the styler
            _, styler, _ = formatted_styler_for(
                sub_table,
                depths=depth,
                # Select the number of nodes for MIS, only.
                agg_values=values,
                with_fancy_values=True,
                cmap=(  # pyright: ignore[reportArgumentType]
                    table_cmap_num_instances
                    if values == "num_instances"
                    else table_cmap_performance
                )
                if isinstance(values, str)
                else table_cmap_combined,
                precision={performance_metric: 1, "num_instances": 0},
                missing_data_str="-",
                target_format="latex",
                show_empty_rows=True,
            )

            # Save table to separate LaTeX file
            with open(_latex_filename, "w") as f_table:
                _ = f_table.write(generated_on_str)
                _ = f_table.write(
                    "% Data is {value_label} for depth P={depth}.\n".format(
                        value_label=value_label, depth=depth
                    )
                )
                f_table.write(
                    convert_evaluation_to_multicolumn_latex(
                        styler.to_latex(
                            # f_table,
                            convert_css=True,
                            hrules=True,
                            clines="skip-last;data",
                        ),
                        num_metrics=len(values) if isinstance(values, list) else 1,
                    )
                )
            _ = f.write(
                r"""
\begin{{table}}[H]
    \centering
    \input{{{filename}}}
    \caption{{\textbf{{{value_label} for $P={depth}$.}} Graph types are Erdos Renyi (ER), Heavy-Hex (HH), Line-to-Full (LB), and Random Regular (RR).{additional_caption}}}
\end{{table}}
""".format(
                    filename=_latex_filename,
                    depth=depth,
                    value_label=value_label,
                    additional_caption=additional_caption,
                )
            )
            _ = f.write(r"\clearpage")
        # _ = f.write(r"\clearpage")

## Compiling and previewing all tables

Tables are written to `table_p<depth>_<metric>.tex` where `<depth>` is the QAOA
depth and `<metric>` is either `approximation_ratio` or `num_instances`. These
LaTeX files contain the `tabular` environments to include the given tables.
Numerical values are formatted with siunitx for easier precision and uncertainty
handling. All of these tables are then included in `tables.tex` as a list of
sections, one per depth. `summary_tables.tex` is an example LaTeX document that
(i) has an appropriate preamble for rendering the tables and (ii) shows how to
include them in a larger document.

To compile the _sample document_ `summary_tables.tex`, run the following command
after generating the LaTeX files:

```bash
latexmk -pdf summary_tables.tex
```


In [20]:
!latexmk -pdf summary_tables.tex

Rc files read (in order):
  NONE
Latexmk: This is Latexmk, John Collins, 9 March 2026. Version 4.88.
Latexmk: applying rule 'pdflatex'...
Rule 'pdflatex':  Reasons for rerun
Changed files or newly in use/created:
  list_of_tables_MIS.tex
  table_MIS_p01_both.tex
  table_MIS_p01_energy.tex
  table_MIS_p01_num_instances.tex
  table_MIS_p02_both.tex
  table_MIS_p02_energy.tex
  table_MIS_p02_num_instances.tex
  table_MIS_p03_both.tex
  table_MIS_p03_energy.tex
  table_MIS_p03_num_instances.tex
  table_MIS_p04_both.tex
  table_MIS_p04_energy.tex
  table_MIS_p04_num_instances.tex
  table_MIS_p05_both.tex
  table_MIS_p05_energy.tex
  table_MIS_p05_num_instances.tex
  table_MIS_p06_both.tex
  table_MIS_p06_energy.tex
  table_MIS_p06_num_instances.tex
  table_MIS_p07_both.tex
  table_MIS_p07_energy.tex
  table_MIS_p07_num_instances.tex
  table_MIS_p08_both.tex
  table_MIS_p08_energy.tex
  table_MIS_p08_num_instances.tex
  table_MIS_p09_both.tex
  table_MIS_p09_energy.tex
  table_MIS_p09_num_in

# Missing Files


## Setup


In [21]:
from IPython.display import display

In [22]:
if table.problem_class == "MC":
    missing_depth = 10
elif table.problem_class == "MIS":
    missing_depth = 6
else:
    raise ValueError(
        f"Problem class {table.problem_class!r} not supported for missing-config generation."
    )


## Identify all methods we want to include in the table


In [23]:
# These are the methods we would like to include in our tables. Some are
# 'virtual' _no_opt methods as no method JSON file exists, but we extract the
# data from an intermediate run of an _opt-method run.
def _filter_configs_for_problem_class(
    configs: list[MethodConfigJSON], problem_class: str
) -> list[MethodConfigJSON]:
    """Filter target instances based on the problem class.

    We remove Fixed Angle results for MIS as we don't want to hardcode a degree.
    MAXCUT tables are fine, as we have varied number of runs per graph type and
    method.
    """
    if problem_class == "MIS":
        # Remove Fixed Angles
        return [m for m in configs if not m.startswith("FA_")]
    return configs


target_methods: dict[
    EvaluationType, list[MethodConfigJSON] | dict[bool, list[MethodConfigJSON]]
] = {
    "MPS": {
        True: _filter_configs_for_problem_class(
            cast(
                list[MethodConfigJSON],
                [
                    "FA_MPSAer_no_opt.json",
                    "FA_MPSAer_opt.json",
                    "F_MPSAer.json",
                    "I_MPSAer.json",
                    "LR_MPSAer_angle_opt.json",
                    "LR_MPSAer_opt.json",
                    "RTS_MPSAer.json",
                    "TQA_MPSAer_no_opt.json",
                    "TQA_MPSAer_opt.json",
                ],
            ),
            table.problem_class,
        ),
        False: _filter_configs_for_problem_class(
            cast(
                list[MethodConfigJSON],
                [
                    "FA_MPS_no_opt.json",
                    "FA_MPS_opt.json",
                    "F_MPS.json",
                    "I_MPS.json",
                    "LR_MPS_angle_opt.json",
                    "LR_MPS_opt.json",
                    "RTS_MPS.json",
                    "TQA_MPS_no_opt.json",
                    "TQA_MPS_opt.json",
                ],
            ),
            table.problem_class,
        ),
    },
    "PP": _filter_configs_for_problem_class(
        cast(
            list[MethodConfigJSON],
            [
                "FA_PP_no_opt.json",
                "FA_PP_opt.json",
                "F_PP.json",
                "I_PP.json",
                "LR_PP_angle_opt.json",
                "LR_PP_opt.json",
                # We don't want Parameter Transfer at all.
                # "PT_PP_AAAM.json",
                "RTS_PP.json",
                "TQA_PP_no_opt.json",
                "TQA_PP_opt.json",
            ],
        ),
        table.problem_class,
    ),
    "SV": _filter_configs_for_problem_class(
        cast(
            list[MethodConfigJSON],
            [
                "FA_SV_no_opt.json",
                "FA_SV_opt.json",
                "F_SV.json",
                "I_SV.json",
                "LR_SV_angle_opt.json",
                "LR_SV_opt.json",
                "TQA_SV_no_opt.json",
                "TQA_SV_opt.json",
                "TS_SV.json",
            ],
        ),
        table.problem_class,
    ),
}

all_methods: list[MethodConfigJSON] = list(
    set(
        method
        for methods in target_methods.values()
        for method in (
            methods
            if isinstance(methods, list)
            else [m for sublist in methods.values() for m in sublist]
        )
    )
)

### Create Row Index for Pivot Tables


In [24]:
# Create multi-index of target evaluation and method labels.
row_index = pd.MultiIndex.from_tuples(
    [
        (
            evaluation_label,
            method_label,
        )
        for evaluation_label, method_label in product(
            ["MPS (Aer)", "MPS (Quimb)", "PP", "SV"],
            sorted(
                set(
                    [
                        utils.labels.trainer_config_to_method_label(m)
                        for m in all_methods
                    ]
                )
            ),
        )
    ],
)

## Filter table for reference methods (MAXCUT only)


In [25]:
# Filter for common instances for MAXCUT P=10 table
if table.problem_class == "MC":
    assert REFERENCE_METHODS_10 is not None
    # We filter by reference methods as table was not filtered by common
    # instances. Only sub_table during LaTeX table generation is filtered, and
    # only for P=10.
    missing_table = table.only_common_instances(
        REFERENCE_METHODS_10,
        per_depth=True,
    )
elif table.problem_class == "MIS":
    # We don't filter by num_nodes as we already did that in the table creation.
    missing_table = table
else:
    raise ValueError(f"Unrecognised problem class {table.problem_class!r}.")

## Pivot Table of Existing Runs


In [26]:
# 1. Show the number of runs
pivot, _, _ = formatted_styler_for(
    missing_table,
    agg_values="num_instances",
    depths=missing_depth,
    show_empty_rows=True,
)
# We need to reprocess the values to floats as formatted_styler_for converts
# them to strings.
pivot = pivot.map(lambda x: float(x))
pivot = pivot.reindex(index=row_index)
styler: Styler = pivot.style.format(na_rep="-", precision=0)  # pyright: ignore[reportAssignmentType]
styler = styler.background_gradient("Greens", axis=None).highlight_null("red")
display(styler)


## Get Missing Instances


### Create Mapping of Target Instances


In [27]:
# Helper functions for saving and loading the target instances to a JSON file.
def __sort_graph_key(graph_key: GraphKey) -> str:
    # The first 4 characters of graph_key are the graph indexes. We move this to
    # the end of the sorting key so we always group (1) the same sized graphs
    # together and (2) graphs of the same type together.
    if len(graph_key) < 4:
        # Catch-all in-case we don't have enough characters. This should never
        # be the case, but we do it anyway.
        return graph_key
    return graph_key[3:] + graph_key[:]


def save_target_instances_json(
    obj: Mapping[EvaluationType, set[GraphKey] | Mapping[bool, set[GraphKey]]], path
):
    output = {}
    for key1, val1 in obj.items():
        if isinstance(val1, set):
            output[key1] = list(sorted(val1, key=__sort_graph_key))
        else:
            output[key1] = {}
            for key2, val2 in val1.items():
                if isinstance(val2, set):
                    output[key1][key2] = list(sorted(val2, key=__sort_graph_key))
                else:
                    output[key1][key2] = val2
    with open(path, "w") as f:
        json.dump(output, f, indent=2)


def load_target_instances_json(
    path,
) -> dict[EvaluationType, set[GraphKey] | dict[bool, set[GraphKey]]]:
    with open(path, "r") as f:
        data = json.load(f)
    output: dict[EvaluationType, set[GraphKey] | dict[bool, set[GraphKey]]] = {}
    key1: EvaluationType
    for key1, val1 in data.items():
        assert isinstance(key1, str)
        if isinstance(val1, list):
            output[key1] = set(val1)
        else:
            output[key1] = {}
            for key2, val2 in val1.items():
                _bool_key2 = key2 == "true"
                if isinstance(val2, list):
                    output[key1][_bool_key2] = set(val2)  # pyright: ignore[reportIndexIssue]
                else:
                    output[key1][_bool_key2] = val2  # pyright: ignore[reportIndexIssue]
    return output


In [28]:
target_instances_for_missing: dict[
    EvaluationType, set[GraphKey] | dict[bool, set[GraphKey]]
] = {}
if missing_table.problem_class == "MC":
    # ========================================
    # Determine the target instances for MAXCUT
    # ========================================
    #
    # This is done in two possible ways:
    # 1. Using `{problem_class}_target_instances.json`.
    # 2. By determining all instances we have in the data and populating the
    #    table with them.
    #
    # Depending on what you are doing, you will choose one of these. Note that
    # we do not use these methods for MIS. Instead we programmatically define
    # those earlier in the notebook.
    #
    # BY DEFAULT YOU SHOULD USE OPTION 1. THIS ENSURES YOU ARE CHECKING THE
    # CORRECT INSTANCES.

    # OPTION 1
    # ========
    # target_instances_for_missing = load_target_instances_json(
    #     f"{table.problem_class}_target_instances.json"
    # )

    # OPTION 2
    # ========
    target_instances_for_missing = {}
    for _inst, _inst_data in missing_table.data.items():
        for trainer_config, trainer_data in _inst_data.items():
            if missing_depth not in trainer_data:
                continue
            evaluation = utils.labels.trainer_config_to_evaluation(trainer_config)
            with_aer = utils.labels.method_uses_aer(trainer_config)
            if evaluation not in target_instances_for_missing:
                if evaluation == "MPS":
                    target_instances_for_missing[evaluation] = {
                        True: set(),
                        False: set(),
                    }
                else:
                    target_instances_for_missing[evaluation] = set()
            if evaluation == "MPS":
                target_instances_for_missing[evaluation][with_aer].add(_inst)  # pyright: ignore[reportIndexIssue]
            else:
                target_instances_for_missing[evaluation].add(_inst)  # pyright: ignore[reportAttributeAccessIssue]

elif missing_table.problem_class == "MIS":
    assert TARGET_INSTANCES is not None
    target_instances_for_missing = TARGET_INSTANCES
else:
    raise ValueError(
        f"Cannot compute target instances for problem class {missing_table.problem_class!r}."
    )

### Load Failed Runs


In [29]:
failed_runs_filename = Path(f"failed_runs_{table.problem_class}.json")
failed_runs: dict[GraphKey, dict[MethodConfigJSON, dict[Depth, str]]]
if failed_runs_filename.exists():
    failed_runs = missing_table.load_failed_configs_from_json(failed_runs_filename)
else:
    failed_runs = defaultdict(lambda: defaultdict(dict))


### Compute Missing Instances Dataframe


In [30]:
missing: dict[
    EvaluationType | tuple[Literal["MPS"], bool],
    dict[MethodConfigJSON, dict[Depth, set[GraphKey]]],
] = missing_table.get_missing_configs(
    target_instances=target_instances_for_missing,
    target_depths=[missing_depth],
    target_methods=target_methods,
    # failed_configs=None,
    failed_configs=failed_runs,
    # We do not return derived configs as we can extract them from `missing`.
    with_derived_configs=False,
)

missing_records = []
for _eval_type, _eval_data in missing.items():
    if _eval_type in ["SV", "PP"]:
        _evaluation = _eval_type
        _with_aer = None
    elif _eval_type[0] == "MPS":
        _evaluation = _eval_type[0]
        _with_aer = _eval_type[1]
    else:
        raise ValueError(f"Evaluation {_eval_type!r} not recognized.")
    for _trainer_config, _method_data in _eval_data.items():
        for _depth, _instances in _method_data.items():
            assert _depth == missing_depth
            missing_records.extend(
                [
                    {
                        "instance": _inst,
                        "method": _trainer_config,
                        "depth": _depth,
                        "evaluation": _evaluation,
                        "with_aer": _with_aer,
                        "method_label": utils.labels.trainer_config_to_method_label(
                            _trainer_config
                        ),
                        "evaluation_label": utils.labels.trainer_config_to_evaluation_label(
                            _trainer_config
                        ),
                        "graph_type": utils.instance.graph_type(_inst),
                    }
                    for _inst in _instances
                ]
            )
df_missing = pd.DataFrame.from_records(missing_records)
df_missing


/home/conrad/Projects/QAOA_Parameter_Setting/QAOA-Parameter-Setting/qaoa_parameter_setting/utils/database/results_database.py:2143: UserWarning: Method 'LR_MPSAer_angle_opt.json' for evaluation 'MPS' (with Aer) has no data in the database
  warnings.warn(
/home/conrad/Projects/QAOA_Parameter_Setting/QAOA-Parameter-Setting/qaoa_parameter_setting/utils/database/results_database.py:2143: UserWarning: Method 'LR_MPSAer_opt.json' for evaluation 'MPS' (with Aer) has no data in the database
  warnings.warn(
/home/conrad/Projects/QAOA_Parameter_Setting/QAOA-Parameter-Setting/qaoa_parameter_setting/utils/database/results_database.py:2143: UserWarning: Method 'RTS_MPSAer.json' for evaluation 'MPS' (with Aer) has no data in the database
  warnings.warn(
/home/conrad/Projects/QAOA_Parameter_Setting/QAOA-Parameter-Setting/qaoa_parameter_setting/utils/database/results_database.py:2143: UserWarning: Method 'F_MPS.json' for evaluation 'MPS' (without Aer) has no data in the database
  warnings.warn(
/h

,instance,method,depth,evaluation,with_aer,method_label,evaluation_label,graph_type
0,007_100nodes_2swap_layers.json,LR_MPSAer_angle_opt.json,6,MPS,True,Linear Ramp*,MPS (Aer),line_to_full
1,003_70nodes_erdosrenyi20percent.json,LR_MPSAer_angle_opt.json,6,MPS,True,Linear Ramp*,MPS (Aer),erdos_renyi
2,000_70nodes_erdosrenyi20percent.json,LR_MPSAer_angle_opt.json,6,MPS,True,Linear Ramp*,MPS (Aer),erdos_renyi
3,005_100nodes_random3regular.json,LR_MPSAer_angle_opt.json,6,MPS,True,Linear Ramp*,MPS (Aer),random_regular
4,002_100nodes_2swap_layers.json,LR_MPSAer_angle_opt.json,6,MPS,True,Linear Ramp*,MPS (Aer),line_to_full
...,...,...,...,...,...,...,...,...
345,006_7_3_heavyhex_144nodes_weighted.json,RTS_PP.json,6,PP,None,Recursive TS,PP,heavy_hex
346,004_70nodes_erdosrenyi20percent.json,RTS_PP.json,6,PP,None,Recursive TS,PP,erdos_renyi
347,005_100nodes_2swap_layers.json,RTS_PP.json,6,PP,None,Recursive TS,PP,line_to_full
348,003_7_3_heavyhex_144nodes_weighted.json,RTS_PP.json,6,PP,None,Recursive TS,PP,heavy_hex


In [31]:
# Save missing instances to CSV.
df_missing[["instance", "method", "depth"]].sort_values(
    ["instance", "method", "depth"]
).to_csv(f"summary_missing_{table.problem_class}.csv")

### Pivot Table of Missing Instances


In [32]:
pivot_missing = df_missing.pivot_table(
    values="instance",
    index=["evaluation_label", "method_label"],
    columns=["graph_type"],
    aggfunc="count",
)
pivot_missing = pivot_missing.reindex(index=row_index)
display(f"Missing Runs for problem_class={table.problem_class} at P={missing_depth}")
display(
    pivot_missing.style.format(precision=0, na_rep=" ")
    .background_gradient("RdYlGn_r")  # pyright: ignore[reportAttributeAccessIssue]
    .highlight_null("transparent")
)

'Missing Runs for problem_class=MIS at P=6'